# Solution D2 · Validation and learning curves: diagnosing a model

**Type**: Interpret

## Objective

Compute a validation curve (depth of a tree) and two learning curves (logistic regression and random forest), and use them
to say whether a model suffers from underfitting or overfitting and whether more data would be needed.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, learning_curve, validation_curve
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

df = ds.loans(with_missing=False)
X = df[["age", "income", "job_tenure", "amount"]].to_numpy()
y = df["defaulted"].to_numpy()
cv = StratifiedKFold(5, shuffle=True, random_state=0)
depths = [1, 2, 3, 5, 8, 12, 20]

## Commented solution

In [ ]:
tr, va = validation_curve(DecisionTreeClassifier(random_state=0), X, y, param_name="max_depth", param_range=depths,
                          cv=cv, scoring="roc_auc")
train_v, val_v = tr.mean(axis=1), va.mean(axis=1)
best_depth = depths[int(np.argmax(val_v))]
gap_v = train_v - val_v
display(pd.DataFrame({"max_depth": depths, "train": train_v, "validation": val_v, "gap": gap_v}).round(3))
print("best depth:", best_depth)

In [ ]:
def classify(i):
    if gap_v[i] > 0.15:
        return "overfitting"
    if val_v[i] < 0.72:
        return "underfitting"
    return "adequate"


diagnosis = {"depth_1": classify(0), "depth_20": classify(len(depths) - 1), "depth_best": classify(depths.index(best_depth))}
print(diagnosis)

In [ ]:
sizes = [0.2, 0.4, 0.6, 0.8, 1.0]
_, _, va_lr = learning_curve(make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)), X, y, train_sizes=sizes, cv=cv, scoring="roc_auc")
_, _, va_rf = learning_curve(RandomForestClassifier(60, min_samples_leaf=5, random_state=0), X, y, train_sizes=sizes, cv=cv, scoring="roc_auc")
val_lr, val_rf = va_lr.mean(axis=1), va_rf.mean(axis=1)
lr_gain = val_lr[-1] - val_lr[2]
rf_gain = val_rf[-1] - val_rf[2]
more_data_needed = {"lr": bool(lr_gain > 0.01), "rf": bool(rf_gain > 0.01)}
print("validation AUC as the data grows - logistic:", val_lr.round(3), "| forest:", val_rf.round(3))
print(f"gain going from 60% to 100%: logistic {lr_gain:+.3f} | forest {rf_gain:+.3f} -> {more_data_needed}")
sufficient_fraction = {name: next(f for f, v in zip(sizes, val) if v >= val[-1] - 0.005)
                       for name, val in [("lr", val_lr), ("rf", val_rf)]}
print("minimum sufficient fraction of data:", sufficient_fraction)

## Check

In [ ]:
with verify():
    check(len(train_v) == 7 and len(val_v) == 7 and best_depth == depths[int(np.argmax(val_v))], "validation curve computed, best_depth consistent", "best_depth = depths[np.argmax(val_v)]")
    check(2 <= best_depth <= 8, "the best depth is intermediate (between 2 and 8): neither too little nor too much", "look at the validation: it goes up and then down")
    check(gap_v[-1] > 0.15 and gap_v[0] < 0.05 and close(gap_v, train_v - val_v), "the gap grows with the depth (tiny at 1, huge at 20)", "gap_v = train_v - val_v")
    check(diagnosis == {"depth_1": "underfitting", "depth_20": "overfitting", "depth_best": "adequate"}, "correct diagnosis for the three trees",
          "depth_1: low validation (underfitting); depth_20: huge gap (overfitting); the best: adequate")
    check(len(val_lr) == 5 and len(val_rf) == 5 and close(lr_gain, val_lr[-1] - val_lr[2]) and close(rf_gain, val_rf[-1] - val_rf[2]),
          "gains computed (100% minus 60% of the data)", "gain = val[-1] - val[2]")
    check(more_data_needed == {"lr": bool(lr_gain > 0.01), "rf": bool(rf_gain > 0.01)} and not more_data_needed["lr"] and not more_data_needed["rf"],
          "with this data neither model gains more than 0.01 by adding data beyond 60%: the limit is not the quantity", "compare the two gains with 0.01")
    check(sufficient_fraction["lr"] < sufficient_fraction["rf"] and val_lr[-1] > val_rf[-1],
          "the logistic needs less data and its final validation is better than the forest's", "the forest needs more data and stays below the logistic")
done()

## What you learned

- **Validation curve**: with depth 1 the tree is too simple (training and validation both low and close: **underfitting**);
  beyond depth 5-8 the training goes toward 1 while the validation drops (**overfitting**: huge gap). The best point is in the middle.
- **Learning curve**: the logistic is already saturated with 20% of the data (0.78 with 240 rows and with 1200): it has a limited capacity, its
  limit is the model and the features, not the quantity of data. The forest needs more data (it reaches 0.767 only from 40%) but stays
  below the logistic at every size: here more data is not the right lever, you need better features or a different model.
- The two curves answer different questions: validation tells **which complexity**, learning tells **whether to invest in data**. They are
  the cheapest way to decide where to spend the next day of work.

## Common pitfall

Interpreting a learning curve with a single seed and no standard deviation. A gain of 0.005 is within the fold noise: decisions
(collecting more data, changing model) must be made on differences that exceed the variability, and with repeated folds when needed.

## Interview question

<details><summary>Your learning curve shows a large and stable gap between training and validation, even with all the data. What do you do?</summary>

A large and stable gap is overfitting that additional data won't solve by themselves: first I reduce the complexity of the model
(regularization, less depth, fewer features, early stopping), then I reevaluate. Only if the gap shrinks as the data grows (the
validation curve still rises) does it make sense to collect more. If training and validation are both low and close, instead, the problem is the
model being too simple or the features being barely informative.

</details>